# Tutorial: Synthetic preprocessing and provenance contract

**Audience:** CytoBridge users preparing a count matrix for spatial training.

**Prerequisites:** Python 3.10+, basic AnnData familiarity, and, from the repository root, `python -m pip install -e '.[preprocess]'`. Use a Jupyter frontend from the same environment.

**Learning goals:** build a minimal raw-count AnnData object, run the public preprocessing API exactly once, inspect the recorded contract, and recognize the double-transformation guard. This notebook uses synthetic data only; it contains no manuscript result or benchmark claim.


## Outline

1. Create a deterministic spatial count dataset.
2. Normalize, log-transform, select latent features, and fit PCA.
3. Inspect provenance and deterministic array identities.
4. Exercise the transform guard and required-feature contract.


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from anndata import AnnData

import CytoBridge

SEED = 42
rng = np.random.default_rng(SEED)
environment = {
    "cytobridge": CytoBridge.__version__,
    "cytobridge_source": str(Path(CytoBridge.__file__).resolve()),
    "python": platform.python_version(),
    "seed": SEED,
}
environment


## Step 1 — Create a small raw-count input

The contract needed here is deliberately narrow: non-negative integer-like counts, an explicit time column, cell-type labels, and two spatial coordinates. Real projects should replace this cell with a hash-verified loader, not paste dataset-specific absolute paths into the notebook.


In [ ]:
n_cells, n_genes = 72, 40
stages = np.repeat(np.array(["E0", "E1", "E2"]), n_cells // 3)
counts = rng.poisson(2.0, size=(n_cells, n_genes)).astype(np.float32)
counts[stages == "E1", :5] += 2
counts[stages == "E2", 5:10] += 3
counts[:, 39] = 0  # deterministic low-information marker for the required-feature exercise
spatial = np.column_stack([
    np.linspace(0.0, 1.0, n_cells),
    rng.normal(0.0, 0.08, size=n_cells),
]).astype(np.float32)

obs = pd.DataFrame(
    {
        "stage": stages,
        "Annotation": np.where(np.arange(n_cells) % 2 == 0, "TypeA", "TypeB"),
    },
    index=[f"Cell{i:03d}" for i in range(n_cells)],
)
var = pd.DataFrame(index=[f"Gene{i:03d}" for i in range(n_genes)])
adata = AnnData(X=counts.copy(), obs=obs, var=var)
adata.layers["counts"] = counts.copy()
adata.obsm["spatial"] = spatial

assert np.isfinite(counts).all() and (counts >= 0).all()
assert np.allclose(counts, np.rint(counts), rtol=0.0, atol=0.0)
assert adata.obsm["spatial"].shape == (n_cells, 2)
adata.obs.groupby(["stage", "Annotation"], observed=True).size().rename("cells")


## Step 2 — Apply the public preprocessing API once

`expression_layer='counts'` makes the raw source explicit. Strict validation checks the full selected source before the standard sequence `normalize_total(target_sum=10000) → log1p`. Highly variable genes drive PCA without subsetting the gene-expression matrix. Required features are added to the PCA mask if necessary.


In [ ]:
processed = CytoBridge.pp.preprocess(
    adata.copy(),
    time_key="stage",
    time_mapping={"E0": 0.0, "E1": 1.0, "E2": 2.0},
    n_top_genes=20,
    n_pcs=8,
    expression_layer="counts",
    raw_count_validation="strict",
    required_latent_features=["Gene000", "Gene039"],
)

{
    "X_shape": processed.X.shape,
    "X_latent_shape": processed.obsm["X_latent"].shape,
    "mapped_times": sorted(processed.obs["time_point_processed"].unique().tolist()),
}


## Step 3 — Inspect provenance instead of inferring it

The processed object records its expression source, validation scope, transform order, resolved time mapping, PCA feature count, and exact PCA fit center. The compact hashes below are tutorial-local identities; they are not cross-environment golden values because dependency versions may change floating-point PCA output. A formal run should pin the environment and also hash the input file, configuration, code commit, and output manifest.


In [ ]:
def array_sha256(value: np.ndarray) -> str:
    array = np.ascontiguousarray(np.asarray(value))
    digest = hashlib.sha256()
    digest.update(array.dtype.str.encode("ascii"))
    digest.update(json.dumps(list(array.shape), separators=(",", ":")).encode("ascii"))
    digest.update(array.tobytes(order="C"))
    return digest.hexdigest()

info = processed.uns["preprocess_info"]
assert info["expression_source"] == "layers['counts']"
assert info["raw_count_validation_effective"] == "strict"
assert info["transformation_sequence"] == ["normalize_total", "log1p"]
assert processed.obsm["X_latent"].shape == (n_cells, 8)
assert processed.var["pca_center"].shape == (n_genes,)
assert all(bool(processed.var.loc[name, "highly_variable"]) for name in ["Gene000", "Gene039"])

contract_summary = {
    "expression_source": info["expression_source"],
    "raw_count_validation": info["raw_count_validation_effective"],
    "transformations": info["transformation_sequence"],
    "n_latent_fit_features": info["n_latent_fit_features"],
    "latent_sha256": array_sha256(processed.obsm["X_latent"]),
    "pca_center_sha256": array_sha256(processed.var["pca_center"].to_numpy()),
}
contract_summary


## Pitfall — Do not normalize and log-transform twice

Passing an already transformed `X` back through the default transform path is rejected when the count layer and preprocessing metadata reveal the conflict. For a clean new run, start again from the raw count layer. A labelled legacy replay must opt in explicitly rather than silently changing expression semantics.


In [ ]:
try:
    CytoBridge.pp.preprocess(
        processed.copy(),
        time_key="stage",
        n_top_genes=20,
        n_pcs=8,
    )
except ValueError as exc:
    message = str(exc)
    assert "double-transform" in message
    print("Guarded as expected:", message.splitlines()[0])
else:
    raise AssertionError("Already transformed X was unexpectedly accepted")


## Exercise — Preserve a required marker outside the top HVGs

Rerun from the untouched raw object with only 10 statistical HVGs and require `Gene039`. Predict whether the gene-level matrix will be subset, then verify the mask and provenance. The answer scaffold is below.


In [ ]:
exercise = CytoBridge.pp.preprocess(
    adata.copy(),
    time_key="stage",
    time_mapping={"E0": 0.0, "E1": 1.0, "E2": 2.0},
    n_top_genes=10,
    n_pcs=5,
    expression_layer="counts",
    raw_count_validation="strict",
    required_latent_features=["Gene039"],
)

answer = {
    "gene_matrix_not_subset": exercise.n_vars == n_genes,
    "required_marker_in_pca_mask": bool(exercise.var.loc["Gene039", "highly_variable"]),
    "required_requested": exercise.uns["preprocess_info"]["required_latent_features_requested"],
    "required_added": exercise.uns["preprocess_info"]["required_latent_features_added"],
}
assert answer["gene_matrix_not_subset"] and answer["required_marker_in_pca_mask"]
assert answer["required_added"] == ["Gene039"]
answer


## Next steps

For a real dataset, replace only the synthetic loader and keep the explicit checks. Record the input file SHA-256, time mapping, expression layer, required latent features, configuration, source commit, and output identities in a run manifest. Spatial alignment and model training are intentionally outside this tutorial; they require dataset-specific landmarks/graph contracts and must not be inferred from these synthetic values.
